# Evaluate the Dolly inverters (and the authors' checkpoint)

Scores three T5-base inverters on all **737 validation prompts** of the Dolly-15k split. The split
is rebuilt with the same seed as in training, so it is the same set the training run evaluated on.

**Models**
- `ours_ep6`: our run at epoch 6, the lowest eval loss. This is the model vec2text kept as "best".
- `ours_ep40`: our run at the end of training (epoch 40).
- `authors`: `jxm/t5-base__llama-7b__one-million-instructions__emb`, the paper's inverter for
  Llama-2-7B, trained on about 1M prompts.

**Decoding settings**
- `greedy`: plain greedy search, the setting used during training.
- `greedy_norep3`: greedy search that never repeats the same 3 tokens.
- `beam4_norep3`: 4-beam search that never repeats the same 3 tokens.

**Inputs:** the `dolly15k-llama2-7b-logprobs` dataset and the output of the
`vec2text-dolly-llama2-t5base` training notebook. Accelerator: one GPU T4, with internet on.

In [ ]:
REPO_URL = "https://github.com/Yae-sine/Language-Model-Inversion.git"
REPO_BRANCH = "main"

REPO_DIR = "/tmp/lmi"
V2T_DIR = f"{REPO_DIR}/vec2text"
NPY_DIR = "/tmp/llama2_dolly15k"
DATASET_DIR = "/tmp/dolly_llama2_inv"
WORK = "/kaggle/working"
EVAL_DIR = f"{WORK}/eval"
AUTHORS_MODEL = "jxm/t5-base__llama-7b__one-million-instructions__emb"

In [ ]:
import glob, json, os, shutil, subprocess, sys, zipfile


def sh(cmd, cwd=None, env=None, log=None):
    # Run a shell command, stream its output into the notebook, fail loudly on error.
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env=env, executable="/bin/bash",
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    f = open(log, "a") if log else None
    for line in p.stdout:
        print(line, end="", flush=True)
        if f:
            f.write(line)
            f.flush()
    rc = p.wait()
    if f:
        f.close()
    if rc != 0:
        raise RuntimeError(f"command failed with exit code {rc}: {cmd}")


sh("nvidia-smi")
sh(f"rm -rf {REPO_DIR} && git clone --depth 1 -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}")
sh(f"git -C {REPO_DIR} log --oneline -1")

## Environment

`transformers` is pinned to 4.45.2. That is the last release that still accepts the
`evaluation_strategy` setting vec2text's `TrainingArguments` uses (removed in 4.46). It is also the
first release that works with `tokenizers` 0.20, which ships Python 3.13 builds; older `tokenizers`
fail to install on Kaggle's Python 3.13. A constraints file keeps Kaggle's own torch build.

`torchvision` is uninstalled. When it is loaded, `datasets` 3.x's torch formatter imports
`torchvision.io.VideoReader`, which recent torchvision removed, so reading the dataset crashes.
vec2text never uses torchvision.

In [ ]:
torch_ver = subprocess.check_output(
    [sys.executable, "-c", "import torch; print(torch.__version__)"], text=True).strip()
with open("/tmp/constraints.txt", "w") as f:
    f.write(f"torch=={torch_ver}\n")
print("keeping torch", torch_ver)

sh('pip install -q -c /tmp/constraints.txt "transformers==4.45.2" "tokenizers==0.20.3" "datasets>=2.20,<4" '
   '"accelerate==0.34.2" "sentence-transformers==3.0.1" evaluate sacrebleu rouge_score nltk '
   'sentencepiece einops tenacity openai')
sh("pip uninstall -y -q torchvision")
sh(f"pip install -q -e {V2T_DIR} --no-deps")
sh("python -c \"import nltk; nltk.download('punkt', quiet=True); nltk.download('punkt_tab', quiet=True)\"")
sh("python -c \"import torch, transformers, datasets, vec2text; print('torch', torch.__version__, '| transformers', transformers.__version__, '| datasets', datasets.__version__, '| cuda', torch.cuda.is_available())\"")

## Inputs

The files are found by searching `/kaggle/input`, so this works whatever the dataset's mount path is,
and whether or not Kaggle unzipped the `.npz`.

The Llama-2 tokenizer and config are copied to `/kaggle/working/meta-llama/Llama-2-7b-hf`. Every
command then runs from `/kaggle/working`, so the model name `meta-llama/Llama-2-7b-hf` resolves to
that local folder, and the gated Hugging Face repo is never contacted.

In [ ]:
def find(name):
    return sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))


os.makedirs(NPY_DIR, exist_ok=True)
npz = [p for p in find("llama2_dolly15k.npz") if os.path.isfile(p) and zipfile.is_zipfile(p)]
if npz:
    print("extracting", npz[0])
    with zipfile.ZipFile(npz[0]) as z:
        for member in ("instructions.npy", "vectors.npy"):
            if not os.path.exists(f"{NPY_DIR}/{member}"):
                z.extract(member, NPY_DIR)
else:  # Kaggle already unzipped it
    for member in ("instructions.npy", "vectors.npy"):
        src = find(member)[0]
        print("linking", src)
        if not os.path.exists(f"{NPY_DIR}/{member}"):
            os.symlink(src, f"{NPY_DIR}/{member}")

tok_dir = os.path.dirname(find("tokenizer.model")[0])
llama_dir = f"{WORK}/meta-llama/Llama-2-7b-hf"
os.makedirs(llama_dir, exist_ok=True)
for name in ("config.json", "generation_config.json", "special_tokens_map.json",
             "tokenizer_config.json", "tokenizer.json", "tokenizer.model"):
    shutil.copy(f"{tok_dir}/{name}", llama_dir)

sh(f"ls -la {NPY_DIR} {llama_dir}")

## Build the training dataset

`build_dolly_llama2_dataset.py` does the following:
- tokenizes each instruction with the repo's own `tokenize_function`
- drops the instructions longer than 64 T5 tokens
- attaches each log-prob vector as the `frozen_embeddings` column
- splits the data 95/5 into train and validation (seed 42)

In [ ]:
if not os.path.exists(DATASET_DIR):
    sh(f"python {V2T_DIR}/experiment_scripts/logits/build_dolly_llama2_dataset.py "
       f"--data_dir {NPY_DIR} --out {DATASET_DIR}", cwd=WORK)
else:
    print("dataset already built:", DATASET_DIR)

## Evaluate

`eval_inverter.py` builds each model with a weightless Llama, the same way as in training. For the
authors' checkpoint, it downloads only the 7 of 89 files that hold the inverter's own weights
(about 1.1 GB) and skips the Llama weights. Every model reads the same precomputed vectors.

The metrics are vec2text's own (`BaseTrainer._text_comparison_metrics`), so they are directly
comparable with the numbers logged during training.

In [ ]:
def one(pattern):
    hits = find(pattern)
    assert hits, f"not found under /kaggle/input: {pattern}"
    return os.path.dirname(hits[0])

ckpt_ep6 = one("checkpoint-2628/model.safetensors")
ckpt_ep40 = one("checkpoint-17520/model.safetensors")
print("epoch 6: ", ckpt_ep6)
print("epoch 40:", ckpt_ep40)

env = dict(os.environ, PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")
sh(f"python {V2T_DIR}/experiment_scripts/logits/eval_inverter.py "
   f"--dataset_dir {DATASET_DIR} --out_dir {EVAL_DIR} --n_examples 25 "
   f"--model ours_ep6={ckpt_ep6} --model ours_ep40={ckpt_ep40} "
   f"--model authors={AUTHORS_MODEL}",
   cwd=WORK, env=env, log=f"{EVAL_DIR}.log")

## Results table

In [ ]:
import pandas as pd

res = json.load(open(f"{EVAL_DIR}/results.json"))
rows = []
for model, r in res.items():
    for dec in ("greedy", "greedy_norep3", "beam4_norep3"):
        m = r[dec]
        rows.append({"model": model, "decoding": dec, "eval_loss": round(r["loss"], 3),
                     "BLEU": round(m["bleu_score"], 2), "BLEU_sem": round(m["bleu_score_sem"], 2),
                     "token_F1": round(100 * m["token_set_f1"], 1),
                     "exact_%": round(100 * m["exact_match"], 1),
                     "ROUGE": round(100 * m["rouge_score"], 1)})
table = pd.DataFrame(rows)
table.to_csv(f"{EVAL_DIR}/summary.csv", index=False)
print(table.to_string(index=False))